# Sign in to Databricks Free Edition, create your first Unity Catalog schema, and query the same table from a notebook and a SQL warehouse

Your three gym teammates spent the last two years on Databricks. You spent them on Snowflake. They keep saying the workspace "feels different from Snowflake or Redshift" and you have not understood what they mean. The vendor's exam guide says Unity Catalog is the governance plane and Delta Lake is the storage layer, but you have not built anything on the platform yet. You have one session to fix that. By the end of this lab, the rest of the cert will land on a real mental model, not a slide deck.

The hands-on work:

- Create a Unity Catalog schema under `workspace.default` named `labrun_platform_tour`.
- Create a managed Delta table called `orders` with three columns.
- Insert two rows from the notebook.
- Query the same three-level name (`workspace.default.labrun_platform_tour.orders`) from the SQL editor against the Starter warehouse.

Confirming the same two rows appear on both surfaces is what proves the notebook compute and the SQL warehouse share one governance plane.

**Time.** Plan for about 45 minutes hands-on. The Starter warehouse cold-start can push you past 50 minutes the first time it wakes up. Budget up to 75 minutes for the session window.

**Cost.** Zero dollars and zero cents. Databricks Free Edition swallows the compute, the metastore is free, and the two rows you insert add up to fewer bytes than this sentence. The thing you are spending here is your daily compute quota, which resets every 24 hours. Treat it like a video-game energy bar.

This lab maps to Databricks DE Associate Domain 1: Databricks Intelligence Platform (~10% of exam weight, provisional).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks and the Databricks SDK. Pinned versions per
# LAB-CREATION-BLUEPRINT.md build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.6.0 databricks-sdk==0.40.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. UC identifiers (schema, table, volume) use
# snake_case under the labrun_ prefix because Unity Catalog identifiers prefer
# snake_case (hyphens force backtick-quoting everywhere downstream). Workspace
# assets that the lab does not create here (jobs, pipelines, git folders) keep
# kebab-case slugs.

import atexit
import getpass
import json
import sys
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "databricks-platform-tour-and-unity-catalog-basics"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[0].slug exactly

CATALOG = "workspace"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "labrun_platform_tour"
LAB_SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"
ORDERS_TABLE_FQN = f"{LAB_SCHEMA_FQN}.orders"

SEED_ROWS = [(1, "CUST-A", 199.99), (2, "CUST-B", 79.50)]

In [ ]:
# NBVAL_SKIP
# Register the session, validate Databricks credentials via the SDK, and
# resolve the Starter SQL warehouse. This cell must succeed before the
# manifest cell runs anything, per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Databricks workspace URL (https://...azuredatabricks.net or .cloud.databricks.com): ").strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT): ").strip()

if not databricks_host or not databricks_token:
    print("Workspace URL and PAT are both required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired. CurrentUser.me() failed:")
    print(f"  {exc}")
    print()
    print("Refresh your workspace URL and PAT, then re-run this cell.")
    raise SystemExit(1)

CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Credentials validated. Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found in this workspace. Free Edition ships with")
    print("a Starter warehouse by default; if it has been deleted, recreate it")
    print("from the SQL > Warehouses page before re-running this cell.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
}


def run_sql(statement, warehouse_id=None, wait_seconds=180):
    """Submit a SQL statement to the warehouse and return the parsed result.

    Polls statement state up to wait_seconds, raises on FAILED or CANCELED,
    returns a dict {columns: [...], rows: [[...]]} on SUCCEEDED.
    """
    wid = warehouse_id or WAREHOUSE_ID
    resp = w.statement_execution.execute_statement(
        warehouse_id=wid,
        statement=statement,
        wait_timeout="50s",
    )
    statement_id = resp.statement_id
    deadline = time.time() + wait_seconds
    while True:
        state = resp.status.state if resp.status else None
        if state == StatementState.SUCCEEDED:
            break
        if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
            err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
            raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement}")
        if time.time() > deadline:
            raise TimeoutError(
                f"SQL did not finish in {wait_seconds}s. Last state: {state}. "
                f"The Starter warehouse may still be waking up; re-run the cell."
            )
        time.sleep(2)
        resp = w.statement_execution.get_statement(statement_id)

    columns = []
    if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
        columns = [c.name for c in resp.manifest.schema.columns]
    rows = []
    if resp.result and resp.result.data_array:
        rows = list(resp.result.data_array)
    return {"columns": columns, "rows": rows}


register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# Manifest is module-level and in reverse-creation order. Free Edition has no
# hourly-billed resources for this lab (serverless compute auto-pauses, the
# Starter warehouse is shared and outside this lab's lifecycle), so no
# entries are flagged critical. Per RESOURCE-SAFETY-SPEC Rule 4 the orphan
# scan blocks execution if any tagged resources from a prior session exist.


CLEANUP_MANIFEST = [
    CleanupResource(
        type="uc_managed_table",
        id=ORDERS_TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {ORDERS_TABLE_FQN}\"",
    ),
    CleanupResource(
        type="uc_schema",
        id=LAB_SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP SCHEMA IF EXISTS {LAB_SCHEMA_FQN} CASCADE\"",
    ),
]


class DatabricksCleanupAdapter:
    """Inline adapter implementing the labrun-checks CleanupAdapter protocol
    against Databricks Unity Catalog via the SQL Statement Execution API.
    Targets uc_managed_table, uc_volume, uc_volume_contents, and uc_schema
    resources for the DE Associate cert track."""

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "uc_volume":
            run_sql(f"DROP VOLUME IF EXISTS {rid}")
        elif rtype == "uc_volume_contents":
            # rid is the FQN of the volume; clear its contents recursively via
            # the Files API. Treat NotFound as already-gone.
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                listing = list(w.files.list_directory_contents(volume_path))
            except Exception:
                return
            for entry in listing:
                try:
                    if entry.is_directory:
                        w.files.delete_directory(entry.path)
                    else:
                        w.files.delete(entry.path)
                except Exception:
                    pass
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE")
        else:
            raise ValueError(f"DatabricksCleanupAdapter: unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "uc_managed_table":
            catalog, schema, table = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                f"AND table_name = '{table}'"
            )
            result = run_sql(sql)
            return len(result["rows"]) > 0
        if rtype == "uc_volume":
            catalog, schema, volume = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.volumes "
                f"WHERE volume_catalog = '{catalog}' AND volume_schema = '{schema}' "
                f"AND volume_name = '{volume}'"
            )
            result = run_sql(sql)
            return len(result["rows"]) > 0
        if rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                listing = list(w.files.list_directory_contents(volume_path))
                return len(listing) > 0
            except Exception:
                return False
        if rtype == "uc_schema":
            catalog, schema = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.schemata "
                f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
            )
            result = run_sql(sql)
            return len(result["rows"]) > 0
        return False

    def tag_scan(self, credentials, lab_slug, region):
        """Return list of Unity Catalog identifiers tagged with the lab slug.

        Queries the per-entity tag views in system.information_schema and
        returns identifiers in dotted-FQN form so they can be matched against
        manifest ids."""
        arns = []
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
            (
                "SELECT catalog_name, schema_name, volume_name FROM system.information_schema.volume_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql)
            except Exception:
                continue
            for row in result["rows"]:
                arns.append(fmt(row))
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    return CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing UC objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources.")
    print("Run the cleanup cell at the bottom of this notebook first, or")
    print("drop them manually with the SQL commands shown by the cleanup")
    print("cell, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Create the lab schema under workspace.default and tag it

Unity Catalog uses three-level naming: `catalog.schema.table`. Free Edition pre-creates one catalog literally named `workspace` with a default schema named `default`. Your lab schema sits under `workspace.default` and is named `labrun_platform_tour`. The literal string `workspace` is the catalog name on Free Edition. Outside Free Edition, customer workspaces typically get a per-customer catalog name; the exam tests both.

Build it in this order:

1. `CREATE SCHEMA IF NOT EXISTS workspace.default.labrun_platform_tour` so the lab is re-runnable.
2. `ALTER SCHEMA workspace.default.labrun_platform_tour SET TAGS ('labrun_lab_slug' = 'databricks-platform-tour-and-unity-catalog-basics')`. The orphan scan and cleanup tag audit both depend on this tag.

The cleanup cell at the bottom of the notebook drops the schema with `CASCADE`, which removes every object inside it. The tag is the breadcrumb that lets a future session find it.

In [ ]:
# NBVAL_SKIP
# Task 1: Create the lab schema and apply the lab tag. The CREATE is
# idempotent so a kernel restart mid-lab does not blow up.

# YOUR CODE: Run CREATE SCHEMA IF NOT EXISTS for LAB_SCHEMA_FQN via run_sql().

# YOUR CODE: Apply the labrun lab tag with ALTER SCHEMA ... SET TAGS
# ('labrun_lab_slug' = LAB_TAG_VALUE).

print(f"Schema created and tagged: {LAB_SCHEMA_FQN}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: The lab schema exists under workspace.default with the
# labrun lab tag. Polls information_schema for up to 15 seconds because tag
# propagation through the metastore is typically sub-second but not
# instantaneous.


def checkpoint_1(session):
    try:
        deadline = time.time() + 15
        schema_seen = False
        tag_seen = False
        while time.time() < deadline:
            schema_sql = (
                "SELECT schema_owner FROM system.information_schema.schemata "
                f"WHERE catalog_name = '{CATALOG}' AND schema_name = '{LAB_SCHEMA}'"
            )
            schema_result = run_sql(schema_sql)
            schema_seen = len(schema_result["rows"]) > 0

            tag_sql = (
                "SELECT tag_value FROM system.information_schema.schema_tags "
                f"WHERE catalog_name = '{CATALOG}' AND schema_name = '{LAB_SCHEMA}' "
                f"AND tag_name = '{LAB_TAG_KEY}'"
            )
            tag_result = run_sql(tag_sql)
            tag_seen = any(row[0] == LAB_TAG_VALUE for row in tag_result["rows"])

            if schema_seen and tag_seen:
                return CheckpointResult(status="pass")
            time.sleep(2)

        if not schema_seen:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Schema {LAB_SCHEMA_FQN} not found in "
                    f"system.information_schema.schemata. Did the CREATE SCHEMA run?"
                ),
            )
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Schema {LAB_SCHEMA_FQN} exists but is missing tag "
                f"{LAB_TAG_KEY}={LAB_TAG_VALUE}. Apply it with ALTER SCHEMA "
                f"{LAB_SCHEMA_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')."
            ),
        )
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint message says either the schema row is missing from `information_schema.schemata` or the tag row is missing from `information_schema.schema_tags`. Two separate SQL statements, two separate failure modes. Read the message before peeking further.

</details>

<details><summary>Hint 2 (direction)</summary>

Two statements. First, `CREATE SCHEMA IF NOT EXISTS workspace.default.labrun_platform_tour`. Second, `ALTER SCHEMA workspace.default.labrun_platform_tour SET TAGS ('labrun_lab_slug' = 'databricks-platform-tour-and-unity-catalog-basics')`. Both run through `run_sql()`. The constants `LAB_SCHEMA_FQN`, `LAB_TAG_KEY`, and `LAB_TAG_VALUE` are already in scope.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
run_sql(f"CREATE SCHEMA IF NOT EXISTS {LAB_SCHEMA_FQN}")
run_sql(
    f"ALTER SCHEMA {LAB_SCHEMA_FQN} "
    f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')"
)
```

</details>

**Wallet check.** Still at $0.00. Two metastore writes against a free serverless warehouse. The daily compute quota has not moved enough to round up to a single DBU.

## Task 2: Create the managed Delta table with three columns

Managed tables in Unity Catalog are tables where Databricks owns the underlying storage. The contract you should care about: `DROP TABLE` on a managed table removes the underlying Parquet files. (Lab 10 demonstrates the opposite contract on external tables on Premium.) For now, build:

1. `CREATE TABLE workspace.default.labrun_platform_tour.orders (order_id BIGINT, customer_id STRING, order_total DECIMAL(10,2)) USING DELTA`.
2. Apply the lab tag: `ALTER TABLE ... SET TAGS ('labrun_lab_slug' = '...')`. The table is independently tag-scannable; the schema tag does NOT automatically propagate to tables inside the schema.

Three columns, in the exact order above. The checkpoint reads the column order and rejects swaps.

In [ ]:
# NBVAL_SKIP
# Task 2: Create the managed Delta table and apply the lab tag.

# YOUR CODE: Run CREATE TABLE IF NOT EXISTS for ORDERS_TABLE_FQN with the
# three columns (order_id BIGINT, customer_id STRING, order_total DECIMAL(10,2))
# USING DELTA via run_sql().

# YOUR CODE: Apply the labrun lab tag to the table with ALTER TABLE ... SET TAGS.

print(f"Table created and tagged: {ORDERS_TABLE_FQN}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Managed Delta table exists with the required three-column
# shape (order_id BIGINT, customer_id STRING, order_total DECIMAL(10,2)) in
# that order, table_type MANAGED, data_source_format DELTA.


def checkpoint_2(session):
    try:
        sql = (
            "SELECT table_type, data_source_format "
            "FROM system.information_schema.tables "
            f"WHERE table_catalog = '{CATALOG}' AND table_schema = '{LAB_SCHEMA}' "
            f"AND table_name = 'orders'"
        )
        result = run_sql(sql)
        if not result["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Table {ORDERS_TABLE_FQN} not found in "
                    f"system.information_schema.tables. Did CREATE TABLE run?"
                ),
            )
        table_type, data_source_format = result["rows"][0]
        if table_type != "MANAGED":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Table {ORDERS_TABLE_FQN} has table_type={table_type!r}; "
                    f"expected MANAGED. Drop the table and re-create it without "
                    f"a LOCATION clause."
                ),
            )
        if (data_source_format or "").upper() != "DELTA":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Table {ORDERS_TABLE_FQN} has data_source_format="
                    f"{data_source_format!r}; expected DELTA. Re-create with "
                    f"USING DELTA."
                ),
            )

        col_sql = (
            "SELECT column_name, data_type, ordinal_position "
            "FROM system.information_schema.columns "
            f"WHERE table_catalog = '{CATALOG}' AND table_schema = '{LAB_SCHEMA}' "
            f"AND table_name = 'orders' ORDER BY ordinal_position"
        )
        col_result = run_sql(col_sql)
        cols = [(row[0], (row[1] or "").upper()) for row in col_result["rows"]]
        expected = [
            ("order_id", "BIGINT"),
            ("customer_id", "STRING"),
            ("order_total", "DECIMAL(10,2)"),
        ]
        for idx, (want_name, want_type) in enumerate(expected):
            if idx >= len(cols):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Table {ORDERS_TABLE_FQN} is missing column at position "
                        f"{idx + 1}: expected {want_name} {want_type}."
                    ),
                )
            got_name, got_type = cols[idx]
            if got_name != want_name or want_type not in got_type:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Column at position {idx + 1} is {got_name!r} {got_type!r}; "
                        f"expected {want_name!r} {want_type!r}. The three columns "
                        f"must be (order_id BIGINT, customer_id STRING, "
                        f"order_total DECIMAL(10,2)) in that order."
                    ),
                )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint reads `information_schema.tables` and `information_schema.columns`. If it complains about table_type the table got created with a LOCATION clause and is external; if it complains about column order, the three columns are present in the wrong sequence. Read the message first.

</details>

<details><summary>Hint 2 (direction)</summary>

The CREATE statement is one line: `CREATE TABLE IF NOT EXISTS workspace.default.labrun_platform_tour.orders (order_id BIGINT, customer_id STRING, order_total DECIMAL(10,2)) USING DELTA`. No `LOCATION`. No partition clause. The column order is order_id first, customer_id second, order_total third.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
run_sql(
    f"CREATE TABLE IF NOT EXISTS {ORDERS_TABLE_FQN} ("
    f"  order_id BIGINT, "
    f"  customer_id STRING, "
    f"  order_total DECIMAL(10,2)"
    f") USING DELTA"
)
run_sql(
    f"ALTER TABLE {ORDERS_TABLE_FQN} "
    f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')"
)
```

</details>

**Wallet check.** Still at $0.00. A managed Delta CREATE writes one tiny `_delta_log/00000000000000000000.json` commit and that is it. No Parquet files yet because no rows. The serverless warehouse is the only thing burning a sliver of quota.

## Task 3: Insert two seed rows from the notebook surface

The two rows are deliberately fixed values so the validator can assert on exact contents:

| order_id | customer_id | order_total |
| --- | --- | --- |
| 1 | CUST-A | 199.99 |
| 2 | CUST-B | 79.50 |

A single multi-row `INSERT INTO ... VALUES (...)` is fine. The validator reads the rows back ordered by `order_id` and asserts on every value, so spelling matters.

In [ ]:
# NBVAL_SKIP
# Task 3: Insert two seed rows into the orders table.

# YOUR CODE: Run INSERT INTO ORDERS_TABLE_FQN VALUES (1, 'CUST-A', 199.99),
# (2, 'CUST-B', 79.50) via run_sql().

result = run_sql(f"SELECT count(*) FROM {ORDERS_TABLE_FQN}")
row_count = int(result["rows"][0][0]) if result["rows"] else 0
print(f"Rows in {ORDERS_TABLE_FQN}: {row_count}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Exactly two seed rows present with the exact values the
# checkpoint expects.


def checkpoint_3(session):
    try:
        count_result = run_sql(f"SELECT count(*) FROM {ORDERS_TABLE_FQN}")
        if not count_result["rows"]:
            return CheckpointResult(
                status="error",
                error_reason=f"SELECT count(*) returned no rows for {ORDERS_TABLE_FQN}.",
            )
        row_count = int(count_result["rows"][0][0])
        if row_count != 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{ORDERS_TABLE_FQN} has {row_count} rows; expected exactly 2. "
                    f"Truncate and re-insert if you over-inserted."
                ),
            )

        rows_result = run_sql(
            f"SELECT order_id, customer_id, order_total "
            f"FROM {ORDERS_TABLE_FQN} ORDER BY order_id"
        )
        actual = []
        for row in rows_result["rows"]:
            try:
                actual.append((int(row[0]), str(row[1]), float(row[2])))
            except (TypeError, ValueError):
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Could not parse row {row!r}. Check column types.",
                )
        expected = [(1, "CUST-A", 199.99), (2, "CUST-B", 79.50)]
        if actual != expected:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Row values do not match. Expected {expected}; got {actual}. "
                    f"Re-run INSERT with exactly these values: "
                    f"(1, 'CUST-A', 199.99), (2, 'CUST-B', 79.50)."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint either says the count is wrong (you under-inserted or double-inserted) or the values are wrong (the row count is right but the contents do not match). The error message names which one.

</details>

<details><summary>Hint 2 (direction)</summary>

One statement: `INSERT INTO workspace.default.labrun_platform_tour.orders VALUES (1, 'CUST-A', 199.99), (2, 'CUST-B', 79.50)`. The constant `ORDERS_TABLE_FQN` already holds the three-level name. If the count is wrong, the cleanest reset is `TRUNCATE TABLE` before re-inserting.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
run_sql(
    f"INSERT INTO {ORDERS_TABLE_FQN} VALUES "
    f"(1, 'CUST-A', 199.99), (2, 'CUST-B', 79.50)"
)
```

If you have already inserted and need to reset:

```python
run_sql(f"TRUNCATE TABLE {ORDERS_TABLE_FQN}")
run_sql(
    f"INSERT INTO {ORDERS_TABLE_FQN} VALUES "
    f"(1, 'CUST-A', 199.99), (2, 'CUST-B', 79.50)"
)
```

</details>

**Wallet check.** Still at $0.00. The INSERT wrote a single tiny Parquet file under the managed table's storage path and one `_delta_log` JSON commit. Free Edition swallows it.

## Task 4: Verify the same two rows from the Starter SQL warehouse

This is the proof. Open the SQL editor in the workspace UI (left nav, SQL > Editor), attach the Starter warehouse, and run:

```sql
SELECT order_id, customer_id, order_total
FROM workspace.default.labrun_platform_tour.orders
ORDER BY order_id;
```

You should see the exact two rows you inserted. Same three-level name, different compute surface, same governance plane. The notebook attached to serverless all-purpose compute and the SQL warehouse are looking at the same physical bytes through Unity Catalog.

The checkpoint below does NOT read your manual SQL editor query. It re-issues the same statement against the warehouse via the Statement Execution API so the validation is independent of what you typed into the UI. The manual SQL editor step is the human-side proof; the checkpoint is the machine-side proof.

In [ ]:
# NBVAL_SKIP
# Task 4: Walk the student through running the verification query via the
# SQL editor manually. The checkpoint below issues the same query via the
# Statement Execution API for independent validation.

print("Step 1. Open the workspace UI in your browser.")
print("Step 2. Click SQL > Editor in the left nav.")
print("Step 3. Pick the Starter warehouse from the dropdown at the top.")
print("Step 4. Paste this query and click Run:")
print()
print(f"    SELECT order_id, customer_id, order_total")
print(f"    FROM {ORDERS_TABLE_FQN}")
print(f"    ORDER BY order_id;")
print()
print("Step 5. Confirm you see two rows: (1, 'CUST-A', 199.99) and (2, 'CUST-B', 79.50).")
print()
print(f"Warehouse resolved by the API: {WAREHOUSE.name} ({WAREHOUSE_ID})")
print("If the warehouse is cold, the first query can take 30 to 90 seconds.")
print("Asking the Starter warehouse to wake up...")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: Same two rows visible from the Starter SQL warehouse via the
# Statement Execution API. Resolves the warehouse, runs the SELECT, and
# asserts the row set independently of any notebook variable. The cold-start
# tolerance comes from the run_sql() helper's 180-second ceiling.


def checkpoint_4(session):
    try:
        warehouses_now = list(w.warehouses.list())
        if not warehouses_now:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "No SQL warehouses found. Free Edition ships with a Starter "
                    "warehouse; recreate it from the SQL > Warehouses page."
                ),
            )
        target = warehouses_now[0]

        count_result = run_sql(
            f"SELECT count(*) FROM {ORDERS_TABLE_FQN}",
            warehouse_id=target.id,
        )
        if not count_result["rows"]:
            return CheckpointResult(
                status="error",
                error_reason="Warehouse SELECT count(*) returned no rows.",
            )
        row_count = int(count_result["rows"][0][0])
        if row_count != 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Warehouse query against {ORDERS_TABLE_FQN} returned "
                    f"{row_count} rows; expected 2. The notebook and the warehouse "
                    f"share one metastore, so a mismatch means Task 3 did not "
                    f"complete successfully."
                ),
            )

        rows_result = run_sql(
            f"SELECT order_id, customer_id, order_total "
            f"FROM {ORDERS_TABLE_FQN} ORDER BY order_id",
            warehouse_id=target.id,
        )
        actual = []
        for row in rows_result["rows"]:
            try:
                actual.append((int(row[0]), str(row[1]), float(row[2])))
            except (TypeError, ValueError):
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Could not parse warehouse row {row!r}.",
                )
        expected = [(1, "CUST-A", 199.99), (2, "CUST-B", 79.50)]
        if actual != expected:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Warehouse rows do not match expected. Expected {expected}; "
                    f"got {actual}."
                ),
            )
        return CheckpointResult(status="pass")
    except TimeoutError as exc:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Starter warehouse did not respond in time: {exc}. "
                f"Cold-start can take 90 seconds; re-run this cell."
            ),
        )
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint either timed out waiting for the warehouse to wake up, or it ran the query and the row set did not match what Task 3 inserted. The error message names which one. Cold-start failures usually clear on a second run.

</details>

<details><summary>Hint 2 (direction)</summary>

No code to write for this task; the checkpoint runs the SELECT against the warehouse for you. If the validator fails with a row mismatch, the issue is back in Task 3 (the notebook insert was wrong); the warehouse and the notebook share one metastore. If the validator fails with a timeout, the Starter warehouse is asleep. Open the SQL editor in the UI, pick the Starter warehouse, and run a `SELECT 1` to wake it up, then re-run this cell.

</details>

<details><summary>Hint 3 (spoiler)</summary>

If the warehouse needs a kick:

```python
result = run_sql("SELECT 1", warehouse_id=WAREHOUSE_ID)
print(result["rows"])
```

If Task 3 needs a redo:

```python
run_sql(f"TRUNCATE TABLE {ORDERS_TABLE_FQN}")
run_sql(
    f"INSERT INTO {ORDERS_TABLE_FQN} VALUES "
    f"(1, 'CUST-A', 199.99), (2, 'CUST-B', 79.50)"
)
```

</details>

**Wallet check.** Still at $0.00. The validator issues two more SQL statements (`SELECT count(*)` and the full row read) against the Starter warehouse. Both are a fraction of a warehouse-second. Free Edition has not noticed.

## Cleanup

Time to tear it all down. The cell below runs through your manifest in reverse-creation order (the orders table first, then the schema with `CASCADE`), then double-checks UC information_schema to confirm both are gone. Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order using the inline Databricks adapter. No critical (hourly-billed)
# resources in this lab, so the canonical summary always reports zero
# critical destructions.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: $0.00.** Databricks Free Edition swallowed the compute, the metastore writes were free, and the Starter warehouse spent a handful of warehouse-seconds across the lab. The orders table and the lab schema were destroyed by the cleanup cell above, so the daily compute quota is the only thing this session touched, and it resets at 00:00 UTC. Your morning coffee cost infinitely more than the Databricks bill.

## Reflection

These are not graded. They are for you.

1. Walk through what Unity Catalog does when you write `INSERT INTO workspace.default.labrun_platform_tour.orders VALUES (1, 'CUST-A', 199.99)` from a notebook and then read the same three-level name from a SQL warehouse. Name each layer the statement traverses (compute, metastore, storage) and what Unity Catalog enforces at each layer.

2. The notebook and the SQL editor are different compute surfaces (serverless all-purpose and a SQL warehouse). Why does Unity Catalog give them the exact same view of the table? What would change if Unity Catalog were absent and you were on the legacy Hive metastore instead?

## Exam-Style Practice

**Question 1 (Domain 1, three-level namespace):**

You run `SELECT count(*) FROM orders` from a Databricks notebook attached to serverless compute and get `[TABLE_OR_VIEW_NOT_FOUND] orders cannot be found`. The table exists at `workspace.default.labrun_platform_tour.orders`. Which statement best explains the error?

A. Unity Catalog requires the full three-level name (`catalog.schema.table`) unless the session has been pinned to a default catalog and schema; the bare name `orders` resolves against the wrong namespace.

B. Serverless compute cannot read Unity Catalog managed tables; it only reads external tables on Free Edition.

C. The user lacks `SELECT` privilege on the table; Unity Catalog returns `TABLE_OR_VIEW_NOT_FOUND` rather than a permissions error for security reasons.

D. The Delta table is missing a transaction log; running `MSCK REPAIR TABLE` on the table will resolve it.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: Unity Catalog uses three-level naming. A bare `orders` resolves against the session's default catalog and schema, which on a fresh notebook is unset; the student needs the full `workspace.default.labrun_platform_tour.orders` or to issue `USE CATALOG workspace; USE SCHEMA labrun_platform_tour;` first.
- B is wrong: serverless compute reads managed Unity Catalog tables; this is the default Free Edition path.
- C is partially relevant but wrong here. Unity Catalog does mask permission errors as `NOT_FOUND` in some contexts, but the underlying issue in the scenario is the missing namespace prefix, not privilege.
- D is wrong: `MSCK REPAIR TABLE` is a legacy Hive command for partition discovery; Delta has no equivalent need and the error is namespace, not corruption.

</details>

**Question 2 (Domain 1, compute service selection):**

A data engineering team is choosing between (a) serverless all-purpose compute and (b) a serverless SQL warehouse for an ad-hoc analyst who runs one to two BI-style SQL queries per hour. The constraint is fast first-query response time and the lowest cost per query. Which service is the better fit?

A. Serverless all-purpose compute, because it runs Python and SQL and has the lowest cost per second.

B. Serverless SQL warehouse, because it is optimized for BI-style SQL with Photon enabled, auto-stops on idle, and has fast first-query response targeted at ad-hoc usage.

C. A classic all-purpose cluster pinned to dc2.large nodes, because classic clusters give the most predictable BI latency.

D. Lakeflow Jobs serverless task compute, because it is the cheapest compute on the platform.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: serverless all-purpose is built for interactive notebook development (Python, R, Scala, SQL mixed) and carries higher startup overhead and per-DBU cost than a SQL warehouse for SQL-only BI workloads.
- B is correct: serverless SQL warehouses are tuned for BI patterns with Photon, share state across users, auto-stop on idle, and give sub-10-second first-query response on warm warehouses. They are the platform's recommended choice for ad-hoc analyst SQL.
- C is wrong: dc2.large is Redshift node terminology; Databricks classic clusters use different node SKUs, and classic clusters are not available on Free Edition. Even outside Free Edition, classic clusters carry longer cold-start than serverless SQL warehouses.
- D is wrong: Lakeflow Jobs serverless task compute is optimized for scheduled batch jobs, not interactive ad-hoc SQL; it cannot serve interactive SQL queries through the SQL editor surface.

</details>